# 🧠 MediScan AI — Brain Tumor MRI Classifier (Google Colab)
### Trains MobileNetV2 on the Kaggle Brain Tumor MRI Dataset in ~15–20 minutes using a free GPU.

**4 Classes:** Glioma · Meningioma · Normal (No Tumor) · Pituitary Tumor

---
### ⚠️ Before running: Make sure you have uploaded the dataset zip to Google Drive
- The zip file from Kaggle should be in your **My Drive** root
- Default expected name: `archive.zip` (rename if different in Step 2)

## Step 1 — Check GPU
Run this first to confirm Colab gave you a GPU. You should see something like **Tesla T4**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  No GPU detected! Go to Runtime > Change runtime type > GPU and re-run.")

## Step 2 — Mount Google Drive
This connects Colab to your Google Drive so it can read the dataset zip and save the trained model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

## Step 3 — Unzip the Dataset
Change `ZIP_NAME` below if your zip file has a different name in Google Drive.

In [ ]:
import zipfile, os

# ── CHANGE THIS if your zip file has a different name in Google Drive ──
ZIP_NAME = "archive.zip"

ZIP_PATH    = f"/content/drive/MyDrive/{ZIP_NAME}"
EXTRACT_DIR = "/content/brain_tumor_mri"

print(f"Unzipping {ZIP_PATH} ...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)
print("Done.")

# Show what was extracted
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, "").count(os.sep)
    if level <= 2:
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)")

## Step 4 — Set Dataset Path
The script auto-detects whether the zip extracted with a subfolder or directly into `Training/Testing`.

In [ ]:
# Auto-detect the correct data root (handles extra subfolder from some Kaggle zips)
DATA_ROOT = EXTRACT_DIR
if not os.path.isdir(os.path.join(DATA_ROOT, "Training")):
    # Try one level deeper
    subdirs = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))]
    if subdirs:
        DATA_ROOT = os.path.join(DATA_ROOT, subdirs[0])

print("Data root:", DATA_ROOT)
print("Training folder exists:", os.path.isdir(os.path.join(DATA_ROOT, "Training")))
print("Testing  folder exists:", os.path.isdir(os.path.join(DATA_ROOT, "Testing")))

## Step 5 — Train the Model
This runs the full two-phase training. On a T4 GPU this takes **~15–20 minutes**.

In [ ]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms
from tqdm.notebook import tqdm   # Colab-friendly progress bars

# ── Config ────────────────────────────────────────────────────────────────
NUM_CLASSES   = 4
EPOCHS        = 15        # total epochs
HEAD_EPOCHS   = 3         # Phase 1 (frozen backbone)
BATCH_SIZE    = 64
LR_HEAD       = 1e-3      # Phase 1 learning rate
LR_FINETUNE   = 1e-4      # Phase 2 learning rate
OUTPUT_PATH   = "/content/drive/MyDrive/mri_classifier.pt"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Transforms ───────────────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Datasets ─────────────────────────────────────────────────────────────
train_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, "Training"), transform=train_tf)
test_ds  = datasets.ImageFolder(os.path.join(DATA_ROOT, "Testing"),  transform=val_tf)
print(f"Train: {len(train_ds)} | Test: {len(test_ds)}")
print(f"Classes: {train_ds.class_to_idx}")

# Weighted sampler to handle class imbalance
labels   = [lbl for _, lbl in train_ds.samples]
counts   = [labels.count(i) for i in range(NUM_CLASSES)]
weights  = [1.0 / counts[lbl] for lbl in labels]
sampler  = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# ── Model — MobileNetV2 ───────────────────────────────────────────────────
def build_model(freeze=True):
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    for p in m.parameters():
        p.requires_grad = not freeze
    in_f = m.classifier[1].in_features  # 1280
    m.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_f, NUM_CLASSES))
    return m

# ── Training helpers ──────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = correct = total = 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(device), labels.to(device)
            if is_train:
                optimizer.zero_grad()
            out  = model(images)
            loss = criterion(out, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += images.size(0)
    return total_loss / total, correct / total

# ── Phase 1: head-only ────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"Phase 1 — head only ({HEAD_EPOCHS} epochs)")
print('='*55)

model = build_model(freeze=True).to(device)
opt   = torch.optim.Adam(model.classifier.parameters(), lr=LR_HEAD)
sch   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HEAD_EPOCHS)

best_acc, best_state = 0.0, None

for epoch in range(1, HEAD_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, opt)
    vl_loss, vl_acc = run_epoch(model, test_loader)[:2]
    sch.step()
    print(f"[P1] {epoch:>2}/{HEAD_EPOCHS} | loss {tr_loss:.3f}/{vl_loss:.3f} "
          f"| acc {tr_acc*100:.1f}%/{vl_acc*100:.1f}% | {time.time()-t0:.0f}s")
    if vl_acc > best_acc:
        best_acc   = vl_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

# ── Phase 2: full fine-tune ───────────────────────────────────────────────
remaining = EPOCHS - HEAD_EPOCHS
print(f"\n{'='*55}")
print(f"Phase 2 — full fine-tune ({remaining} epochs)")
print('='*55)

for p in model.parameters():
    p.requires_grad = True

opt = torch.optim.Adam(model.parameters(), lr=LR_FINETUNE, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=remaining)

for epoch in range(1, remaining + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, opt)
    vl_loss, vl_acc = run_epoch(model, test_loader)[:2]
    sch.step()
    marker = " ✓ best" if vl_acc > best_acc else ""
    print(f"[P2] {epoch:>2}/{remaining} | loss {tr_loss:.3f}/{vl_loss:.3f} "
          f"| acc {tr_acc*100:.1f}%/{vl_acc*100:.1f}% | {time.time()-t0:.0f}s{marker}")
    if vl_acc > best_acc:
        best_acc   = vl_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

print(f"\n✅ Best validation accuracy: {best_acc*100:.2f}%")

## Step 6 — Evaluate & Save Model
Loads the best weights, prints a full classification report, then saves `mri_classifier.pt` to your **Google Drive**.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

CLASS_LABELS = ["Glioma", "Meningioma", "Normal (No Tumor)", "Pituitary Tumor"]

# Load best weights and run final evaluation
model.load_state_dict(best_state)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Final eval"):
        images = images.to(device)
        preds  = model(images).argmax(1).cpu().tolist()
        all_preds  .extend(preds)
        all_labels .extend(labels.tolist())

print("\n📊 Classification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_LABELS))

print("🔢 Confusion Matrix:")
cm = confusion_matrix(all_labels, all_preds)
print(cm)

# Save best model weights to Google Drive
torch.save(best_state, OUTPUT_PATH)
print(f"\n💾 Model saved to Google Drive: {OUTPUT_PATH}")
print("👉 Download it from Drive and place it at:  D:\\FYP\\backend\\models\\mri_classifier.pt")

## ✅ Done! What to do next on your PC

1. Go to **Google Drive** → find `mri_classifier.pt` in your Drive root
2. Download it to your PC
3. Move/copy it to: `D:\FYP\backend\models\mri_classifier.pt`
4. Open a terminal in VS Code and restart the backend:
```powershell
cd D:\FYP\backend
d:\FYP\.venv\Scripts\uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
```
5. The server will print: **"Loaded fine-tuned MRI weights"** — you're done! 🎉

---
---

# 🧪 TEST ONLY — Run this section independently (no retraining needed)
### Use this if you already have `mri_classifier.pt` saved in Google Drive.
Steps below: mount Drive → unzip dataset → load model → full evaluation.


In [ ]:
# ── T1: Mount Drive & set paths ──────────────────────────────────────────
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

MODEL_PATH  = "/content/drive/MyDrive/mri_classifier.pt"   # where you saved it
ZIP_NAME    = "archive.zip"                                  # your dataset zip name
EXTRACT_DIR = "/content/brain_tumor_mri"
ZIP_PATH    = f"/content/drive/MyDrive/{ZIP_NAME}"

# Unzip only if not already done
if not os.path.isdir(EXTRACT_DIR):
    print(f"Unzipping {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    print("Done.")
else:
    print("Dataset already extracted.")

# Auto-detect data root
DATA_ROOT = EXTRACT_DIR
if not os.path.isdir(os.path.join(DATA_ROOT, "Testing")):
    subdirs = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))]
    if subdirs:
        DATA_ROOT = os.path.join(DATA_ROOT, subdirs[0])

print("Data root  :", DATA_ROOT)
print("Model file :", MODEL_PATH)
print("Model exists:", os.path.isfile(MODEL_PATH))


In [ ]:
# ── T2: Load model ───────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torchvision import models

NUM_CLASSES = 4
CLASS_LABELS = ["Glioma", "Meningioma", "Normal (No Tumor)", "Pituitary Tumor"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

def build_model():
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_f, NUM_CLASSES))
    return m

model = build_model()
state_dict = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print("✅ Model loaded successfully")


In [ ]:
# ── T3: Full evaluation on ALL test images ───────────────────────────────
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.notebook import tqdm
import numpy as np

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_ds     = datasets.ImageFolder(os.path.join(DATA_ROOT, "Testing"), transform=val_tf)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

print(f"Test images  : {len(test_ds)}")
print(f"Class mapping: {test_ds.class_to_idx}")

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating"):
        images = images.to(device)
        preds  = model(images).argmax(1).cpu().tolist()
        all_preds .extend(preds)
        all_labels.extend(labels.tolist())

print("\n📊 Classification Report (per-class accuracy):")
print(classification_report(all_labels, all_preds, target_names=CLASS_LABELS, digits=3))

print("🔢 Confusion Matrix (rows=Actual, cols=Predicted):")
cm = confusion_matrix(all_labels, all_preds)
print(f"{'':20}", "  ".join(f"{c[:6]:>6}" for c in CLASS_LABELS))
for i, row in enumerate(cm):
    print(f"{CLASS_LABELS[i][:20]:20}", "  ".join(f"{v:>6}" for v in row))

overall_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"\n✅ Overall Accuracy: {overall_acc*100:.2f}%")


In [ ]:
# ── T4: Show sample wrong predictions ─────────────────────────────────────
import glob, random
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch.nn.functional as F

FOLDER_TO_CLASS = {v: k for k, v in test_ds.class_to_idx.items()}
# e.g. {0: 'glioma', 1: 'meningioma', 2: 'notumor', 3: 'pituitary'}

# Collect wrong predictions
wrong = []
all_files = [path for path, _ in test_ds.samples]
for path, true_idx, pred_idx in zip(all_files, all_labels, all_preds):
    if true_idx != pred_idx:
        wrong.append((path, true_idx, pred_idx))

print(f"Total wrong: {len(wrong)} / {len(all_labels)}")
print(f"Accuracy   : {(1 - len(wrong)/len(all_labels))*100:.1f}%\n")

# Show up to 12 wrong predictions as a grid
sample = wrong[:12]
if sample:
    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    axes = axes.flatten()
    for ax, (path, true_idx, pred_idx) in zip(axes, sample):
        img = Image.open(path).convert("RGB")
        ax.imshow(img, cmap="gray")
        ax.set_title(
            f"True: {CLASS_LABELS[true_idx]}\nPred: {CLASS_LABELS[pred_idx]}",
            fontsize=8, color="red"
        )
        ax.axis("off")
    for ax in axes[len(sample):]:
        ax.axis("off")
    plt.suptitle("❌ Wrong Predictions (sample)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("🎉 No wrong predictions!")
